# Internal link audit + topic clustering with machine learning (2026 refresh)

Companion notebook for the MLforSEO guide **"How to do an SEO internal link audit and topic clustering using machine learning"** by Lazarina Stoy.

What it does, in order:

1. **Loads your crawl**: a Screaming Frog page-content export plus the *All Inlinks* export (or a small demo dataset built from mlforseo.com, so you can run it with no files).
2. **Link audit metrics**: inlinks, unique inlinks, in-content inlinks, orphan pages, broken and nofollow internal links, UTM-tagged internal links.
3. **Anchor text analysis**: generic anchors ("click here", "read more"), branded vs non-branded vs mixed, anchor length, the most common bigrams and trigrams.
4. **Embeddings**: turns every page into a vector with an open sentence-transformers model (or loads embeddings you already exported from Screaming Frog), and flags possible duplicate pages.
5. **Topic clustering**: groups pages into topic clusters, names each cluster with c-TF-IDF keywords and suggests a pillar page.
6. **Bridge pages**: pages that sit between two clusters (your inter-cluster links).
7. **Internal link opportunities**: semantically similar page pairs that are *not linked yet*, with anchor text suggestions found in the source page's own text.
8. **Exports** everything to CSV.

Runs in Google Colab on the free CPU runtime. No API keys needed.

In [ ]:
# 1. Install (Colab already has pandas, numpy, scikit-learn and matplotlib)
import sys
if "google.colab" in sys.modules:
    !pip install -q sentence-transformers beautifulsoup4

## 2. Settings

* `DATA_SOURCE = "demo"` builds a small dataset from the public mlforseo.com blog (about 70 posts) through the WordPress REST API.
* `DATA_SOURCE = "screaming_frog"` uses your own crawl. You need two exports from the Screaming Frog SEO Spider:
  * **Page content**: set up a custom extraction for the main content (Configuration > Custom > Custom Extraction, *Extract Text*), crawl, then export the **Custom Extraction** tab. You need the `Address` column and your extractor column (for example `Content 1`). Adding the `Title 1` column from the Internal tab helps.
  * **Links**: *Bulk Export > Links > All Inlinks*.
* `BRAND_TERMS`: words that make an anchor "branded".

In [ ]:
DATA_SOURCE = "demo"          # "demo" or "screaming_frog"

# --- Screaming Frog files (only used when DATA_SOURCE = "screaming_frog") ---
SF_PAGES_CSV   = "custom_extraction_all.csv"   # Address + your content extractor column
SF_CONTENT_COL = "Content 1"                    # the name of your extractor column
SF_TITLE_COL   = "Title 1"                      # optional; leave as is if missing
SF_INLINKS_CSV = "all_inlinks.csv"              # Bulk Export > Links > All Inlinks
SF_EMBEDDINGS_COL = None   # e.g. "Embeddings 1" if you exported embeddings from the SEO Spider

# --- Demo site ---
DEMO_SITE = "https://www.mlforseo.com"

# --- Audit settings ---
BRAND_TERMS = ["mlforseo", "ml for seo", "lazarina"]
GENERIC_ANCHORS = ["click here", "read more", "here", "this", "learn more", "more",
                   "this article", "this post", "link", "find out more", "continue reading"]

# --- ML settings ---
MODEL_NAME = "BAAI/bge-small-en-v1.5"   # small, fast, strong English embedding model
MAX_WORDS = 350          # words per page sent to the model (title + start of the content)
CLUSTER_METHOD = "kmeans_auto"   # "kmeans_auto" (picks the number of clusters for you),
                                 # "agglomerative" (you set N_CLUSTERS) or "hdbscan" (allows outliers)
N_CLUSTERS = 12                  # agglomerative only
MIN_CLUSTER_SIZE = 3             # hdbscan only
SIMILARITY_PERCENTILE = 90       # suggest links only between pairs in the top 10% most similar pairs on the site
DUPLICATE_THRESHOLD = 0.95       # pairs above this are flagged as possible duplicates, not link opportunities
SUGGESTIONS_PER_PAGE = 5

## 3. Load the data

In [ ]:
import re, html, json, time
from urllib.parse import urlparse, urlunparse
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

def normalise_url(u):
    # lower-case host, drop query string and fragment, add a trailing slash
    p = urlparse(str(u).strip())
    path = p.path if p.path.endswith("/") or "." in p.path.rsplit("/", 1)[-1] else p.path + "/"
    return urlunparse((p.scheme or "https", p.netloc.lower(), path, "", "", ""))

def load_demo(site):
    # Fetch published posts from a WordPress site and pull the links out of each post body
    posts, page = [], 1
    while True:
        r = requests.get(f"{site}/wp-json/wp/v2/posts",
                         params={"per_page": 100, "page": page, "_fields": "link,title,content"},
                         headers={"User-Agent": "Mozilla/5.0 (MLforSEO internal link audit notebook)"},
                         timeout=60)
        if r.status_code != 200 or not r.json():
            break
        posts += r.json()
        if page >= int(r.headers.get("X-WP-TotalPages", 1)):
            break
        page += 1
    host = urlparse(site).netloc.lower()
    page_rows, link_rows = [], []
    for p in posts:
        url = normalise_url(p["link"])
        soup = BeautifulSoup(p["content"]["rendered"], "html.parser")
        for tag in soup(["script", "style", "pre", "code"]):
            tag.decompose()
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if href.startswith("/"):
                href = site + href
            if urlparse(href).netloc.lower() != host:
                continue
            link_rows.append({"Source": url, "Destination": href,
                              "Anchor": a.get_text(" ", strip=True),
                              "Follow": "false" if "nofollow" in (a.get("rel") or []) else "true",
                              "Link Position": "Content", "Status Code": None})
        page_rows.append({"url": url,
                          "title": html.unescape(p["title"]["rendered"]),
                          "content": soup.get_text(" ", strip=True)})
    return pd.DataFrame(page_rows), pd.DataFrame(link_rows)

def load_screaming_frog():
    pages = pd.read_csv(SF_PAGES_CSV)
    pages = pages.rename(columns={"Address": "url", SF_CONTENT_COL: "content"})
    pages["title"] = pages[SF_TITLE_COL] if SF_TITLE_COL in pages.columns else ""
    if SF_EMBEDDINGS_COL and SF_EMBEDDINGS_COL in pages.columns:
        pages["sf_embedding"] = pages[SF_EMBEDDINGS_COL]
    pages["url"] = pages["url"].map(normalise_url)
    links = pd.read_csv(SF_INLINKS_CSV, low_memory=False)
    if "Type" in links.columns:                       # keep hyperlinks only (not images, CSS, JS)
        links = links[links["Type"].astype(str).str.lower().eq("hyperlink")]
    for col in ["Anchor", "Follow", "Link Position", "Status Code"]:
        if col not in links.columns:
            links[col] = None
    return pages, links[["Source", "Destination", "Anchor", "Follow", "Link Position", "Status Code"]].copy()

pages, links = load_demo(DEMO_SITE) if DATA_SOURCE == "demo" else load_screaming_frog()

pages["content"] = pages["content"].fillna("").astype(str)
pages["title"] = pages["title"].fillna("").astype(str)
pages = pages[pages["content"].str.split().str.len() >= 50].drop_duplicates("url").reset_index(drop=True)
links["Anchor"] = links["Anchor"].fillna("").astype(str)
links["raw_destination"] = links["Destination"].astype(str)
links["Source"] = links["Source"].map(normalise_url)
links["Destination"] = links["Destination"].map(normalise_url)
print(f"{len(pages)} pages with content, {len(links)} internal links")
pages.head()

## 4. Link audit metrics

The audit table from the guide, calculated per page: link frequency (inlinks, unique inlinks, orphans) and the common mistakes (broken, nofollow and UTM-tagged internal links).

In [ ]:
crawled = set(pages["url"])
to_pages = links[links["Destination"].isin(crawled)]
content_links = to_pages[to_pages["Link Position"].astype(str).str.lower().eq("content")]

metrics = pages[["url", "title"]].copy()
metrics["inlinks"] = metrics["url"].map(to_pages.groupby("Destination").size()).fillna(0).astype(int)
metrics["unique_inlinks"] = metrics["url"].map(to_pages.groupby("Destination")["Source"].nunique()).fillna(0).astype(int)
metrics["content_inlinks"] = metrics["url"].map(content_links.groupby("Destination")["Source"].nunique()).fillna(0).astype(int)
metrics["outlinks"] = metrics["url"].map(links.groupby("Source")["Destination"].nunique()).fillna(0).astype(int)
metrics["orphan"] = metrics["unique_inlinks"].eq(0)
metrics["no_content_inlinks"] = metrics["content_inlinks"].eq(0)

status = pd.to_numeric(links["Status Code"], errors="coerce")
mistakes = {
    "broken internal links (4xx/5xx)": int((status >= 400).sum()),
    "nofollow internal links": int(links["Follow"].astype(str).str.lower().eq("false").sum()),
    "internal links with UTM parameters": int(links["raw_destination"].str.contains(r"[?&]utm_", case=False).sum()),
    "orphan pages (0 inlinks from crawled pages)": int(metrics["orphan"].sum()),
    "pages with no in-content inlinks": int(metrics["no_content_inlinks"].sum()),
}
print(pd.Series(mistakes).to_string())
print("\nAverage unique inlinks per page:", round(metrics["unique_inlinks"].mean(), 1))
metrics.sort_values("unique_inlinks", ascending=False).head(10)

## 5. Anchor text analysis

A simple, transparent rule-based classifier (no model needed): **generic**, **branded**, **mixed** (brand + topic words), **exact title** (anchor is the target page's title) and **descriptive** (everything else). Then the most frequent bigrams and trigrams across all anchors.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

titles = dict(zip(pages["url"], pages["title"].str.lower().str.strip()))

def classify_anchor(anchor, destination):
    a = re.sub(r"\s+", " ", anchor.lower()).strip(" .:!?→")
    if not a:
        return "empty (image or icon link)"
    if a in GENERIC_ANCHORS or a.startswith(("http://", "https://", "www.")):
        return "generic / naked URL"
    has_brand = any(b in a for b in BRAND_TERMS)
    other_words = [w for w in re.findall(r"[a-z0-9]+", a) if w not in ENGLISH_STOP_WORDS
                   and not any(w in b.split() for b in BRAND_TERMS)]
    if has_brand and other_words:
        return "mixed (brand + topic)"
    if has_brand:
        return "branded"
    if titles.get(destination) and a == titles[destination]:
        return "exact title"
    return "descriptive"

links["anchor_type"] = [classify_anchor(a, d) for a, d in zip(links["Anchor"], links["Destination"])]
links["anchor_words"] = links["Anchor"].str.split().str.len().fillna(0).astype(int)

print(links["anchor_type"].value_counts().to_string())
print("\nMedian anchor length (words):", links["anchor_words"].median())

anchors = links.loc[links["anchor_words"] > 1, "Anchor"].str.lower()
if len(anchors) > 5:
    cv = CountVectorizer(ngram_range=(2, 3), stop_words="english", min_df=2)
    X = cv.fit_transform(anchors)
    ngrams = pd.Series(np.asarray(X.sum(axis=0)).ravel(), index=cv.get_feature_names_out()).sort_values(ascending=False)
    print("\nTop anchor n-grams:\n", ngrams.head(15).to_string())

# pages that only ever get generic anchors are a quick win
generic_only = (links[links["Destination"].isin(crawled)]
                .groupby("Destination")["anchor_type"]
                .agg(lambda s: set(s) <= {"generic / naked URL", "empty (image or icon link)"}))
print("\nPages linked only with generic or empty anchors:", int(generic_only.sum()))

## 6. Embeddings

Each page becomes a vector (a list of numbers that captures its meaning). Pages about similar things end up close together, even when they use different words, which is what LDA and fuzzy matching struggled with.

If you already generated embeddings in the Screaming Frog SEO Spider (v22+, *Config > Content > Embeddings*), set `SF_EMBEDDINGS_COL` above and the notebook uses those instead.

In [ ]:
def page_text(row):
    words = row["content"].split()
    return (row["title"] + ". " + " ".join(words[:MAX_WORDS])).strip()

if "sf_embedding" in pages.columns:
    emb = np.vstack(pages["sf_embedding"].map(lambda s: np.array(json.loads(s) if isinstance(s, str) else s, dtype=float)))
    print("Using embeddings exported from Screaming Frog:", emb.shape)
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME)
    t0 = time.time()
    emb = model.encode([page_text(r) for _, r in pages.iterrows()], batch_size=16,
                       normalize_embeddings=True, show_progress_bar=True)
    print(f"Embedded {len(pages)} pages with {MODEL_NAME} in {time.time() - t0:.0f}s:", emb.shape)

emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)   # unit length, so dot product = cosine similarity
sim = emb @ emb.T

# Similarity scores depend on the model (some models score everything 0.6+), so the link threshold
# is relative to your own site: a pair must be among the top X% most similar pairs.
upper = sim[np.triu_indices(len(sim), 1)]
SIMILARITY_THRESHOLD = float(np.percentile(upper, SIMILARITY_PERCENTILE))
print(f"Median pair similarity {np.median(upper):.2f}; link threshold (p{SIMILARITY_PERCENTILE}) = {SIMILARITY_THRESHOLD:.2f}")

# Near-duplicates: the same check as Screaming Frog's 'Semantically Similar' filter
dups = [(pages.at[i, "title"], pages.at[j, "title"], round(float(sim[i, j]), 3), pages.at[i, "url"], pages.at[j, "url"])
        for i, j in zip(*np.where(np.triu(sim, 1) >= DUPLICATE_THRESHOLD))]
duplicates = pd.DataFrame(dups, columns=["title_a", "title_b", "similarity", "url_a", "url_b"])
print(f"{len(duplicates)} possible duplicate pairs (similarity >= {DUPLICATE_THRESHOLD})")
duplicates.head(10)

## 7. Topic clustering

* **Automatic k-means** (default) tries different numbers of clusters and keeps the one with the best silhouette score (how well each page fits its own cluster compared with the next-nearest one).
* **Agglomerative clustering** builds a tree of pages and cuts it into `N_CLUSTERS` groups.
* **HDBSCAN** finds clusters of different sizes and marks pages that don't fit anywhere as outliers (`-1`), which is useful on big, messy sites.

Each cluster is named with **c-TF-IDF** (the words that are frequent in this cluster but rare in the others, the same idea BERTopic uses) and gets a **pillar candidate**: the page closest to the centre of the cluster, nudged towards pages that already receive more internal links.

In [ ]:
from sklearn.cluster import AgglomerativeClustering, HDBSCAN, KMeans
from sklearn.metrics import silhouette_score

if CLUSTER_METHOD == "hdbscan":
    labels = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean").fit_predict(emb)
elif CLUSTER_METHOD == "agglomerative":
    labels = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric="cosine", linkage="average").fit_predict(emb)
else:
    k_range = range(4, max(5, min(30, len(pages) // 3)))
    scores = {k: silhouette_score(emb, KMeans(k, n_init=10, random_state=42).fit_predict(emb), metric="cosine")
              for k in k_range}
    best_k = max(scores, key=scores.get)
    print(f"Best number of clusters: {best_k} (silhouette {scores[best_k]:.3f})")
    labels = KMeans(best_k, n_init=10, random_state=42).fit_predict(emb)
pages["cluster"] = labels

def ctfidf_labels(texts, labels, top_n=4):
    docs = pd.DataFrame({"text": texts, "cluster": labels}).groupby("cluster")["text"].apply(" ".join)
    cv = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
    tf = cv.fit_transform(docs).toarray().astype(float)
    tf = tf / tf.sum(axis=1, keepdims=True)
    idf = np.log(1 + tf.shape[0] / (1 + (tf > 0).sum(axis=0)))
    scores = tf * idf
    vocab = cv.get_feature_names_out()
    return {c: ", ".join(vocab[np.argsort(-scores[i])[:top_n]]) for i, c in enumerate(docs.index)}

names = ctfidf_labels([page_text(r) for _, r in pages.iterrows()], labels)
pages["cluster_name"] = pages["cluster"].map(names)
pages.loc[pages["cluster"] == -1, "cluster_name"] = "(outlier)"

# centroid similarity + pillar candidate
centroids = {c: emb[labels == c].mean(axis=0) for c in set(labels) if c != -1}
centroids = {c: v / np.linalg.norm(v) for c, v in centroids.items()}
pages["centroid_sim"] = [float(emb[i] @ centroids[c]) if c != -1 else np.nan for i, c in enumerate(labels)]
pages = pages.merge(metrics[["url", "unique_inlinks"]], on="url", how="left")
inl = pages.groupby("cluster")["unique_inlinks"].transform(lambda s: s / s.max() if s.max() else 0)
pages["pillar_score"] = pages["centroid_sim"] * 0.7 + inl * 0.3
pillars = pages[pages["cluster"] != -1].sort_values("pillar_score", ascending=False).groupby("cluster").head(1)
pages["is_pillar_candidate"] = pages["url"].isin(pillars["url"])

summary = (pages.groupby(["cluster", "cluster_name"])
           .agg(pages=("url", "size"), avg_unique_inlinks=("unique_inlinks", "mean"))
           .reset_index()
           .merge(pillars[["cluster", "title"]].rename(columns={"title": "pillar_candidate"}), on="cluster", how="left")
           .sort_values("pages", ascending=False))
print(f"{(summary['cluster'] != -1).sum()} clusters")
summary

### See the clusters

A 2D map of your content. Each dot is a page, coloured by cluster. Pages far from everything else are worth a closer look: they may be off-topic for the site, or a topic you've only written about once.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# t-SNE squeezes the 384-dimensional embeddings into 2D while keeping neighbours close
xy = TSNE(n_components=2, metric="cosine", init="pca", random_state=42,
          perplexity=max(5, min(30, len(pages) // 4))).fit_transform(emb)
big = summary[summary["cluster"] != -1].head(10)["cluster"].tolist()
palette = plt.cm.tab10.colors
fig, ax = plt.subplots(figsize=(11, 7.5))
m_small = ~np.isin(labels, big)
ax.scatter(xy[m_small, 0], xy[m_small, 1], s=35, c="#cfc8dc", label="smaller clusters / outliers")
for k, c in enumerate(big):
    m = labels == c
    ax.scatter(xy[m, 0], xy[m, 1], s=55, alpha=0.9, color=palette[k % 10], label=names.get(c))
    p = pillars[pillars["cluster"] == c]
    if len(p):
        i = pages.index[pages["url"] == p["url"].iloc[0]][0]
        ax.scatter(xy[i, 0], xy[i, 1], s=220, facecolors="none", edgecolors="#1A0B32", linewidths=1.8)
ax.set_title("Topic clusters (circled = pillar candidate)")
ax.set_xticks([]); ax.set_yticks([])
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
plt.tight_layout()
plt.savefig("topic_clusters.png", dpi=150)
plt.show()

## 8. Bridge pages (inter-cluster relationships)

A bridge page is almost as close to a second cluster as it is to its own. These are natural places for links *between* clusters.

In [ ]:
rows = []
cids = sorted(centroids)
if len(cids) > 1:
    C = np.vstack([centroids[c] for c in cids])
    for i, r in pages.iterrows():
        if r["cluster"] == -1:
            continue
        s = emb[i] @ C.T
        order = np.argsort(-s)
        own, other = cids[order[0]], cids[order[1]]
        if own != r["cluster"]:
            other = cids[order[0]]
        gap = float(s[cids.index(r["cluster"])] - s[cids.index(other)])
        rows.append({"url": r["url"], "title": r["title"], "cluster": r["cluster_name"],
                     "also_close_to": names[other], "gap": round(gap, 3)})
bridges = pd.DataFrame(rows).sort_values("gap").head(15) if rows else pd.DataFrame()
bridges

## 9. Internal link opportunities

For every page, the most similar pages it does **not** link to yet. For each suggestion the notebook looks for an anchor text candidate: a phrase from the target page's title or cluster keywords that already appears in the source page's text, so the link can go into an existing sentence.

`priority` puts links to pillar candidates and to pages with no in-content inlinks first.

In [ ]:
existing = set(zip(links["Source"], links["Destination"]))
url_list = pages["url"].tolist()
content_text = pages["content"].tolist()
no_ctx = set(metrics.loc[metrics["no_content_inlinks"], "url"])

def candidate_phrases(title, cluster_name):
    clean = re.sub(r"['\u2019]s\b", "", title.lower())          # Google's -> google
    words = re.findall(r"[a-z0-9\-]+", clean)
    phrases = set()
    for n in (4, 3, 2):
        for k in range(len(words) - n + 1):
            chunk = words[k:k + n]
            if chunk[0] in ENGLISH_STOP_WORDS or chunk[-1] in ENGLISH_STOP_WORDS:
                continue
            phrases.add(" ".join(chunk))
    phrases |= {p.strip() for p in cluster_name.split(",") if len(p.strip().split()) > 1}
    return sorted(phrases, key=lambda p: -len(p.split()))

def find_anchor(src_text, title, cluster_name):
    for p in candidate_phrases(title, cluster_name):
        m = re.search(r"\b" + re.escape(p) + r"\b", src_text, flags=re.IGNORECASE)
        if m:
            start = max(0, m.start() - 70)
            return m.group(0), "..." + src_text[start:m.end() + 70].replace("\n", " ") + "..."
    return "", ""

opps = []
for i, src in enumerate(url_list):
    order = np.argsort(-sim[i])
    found = 0
    for j in order:
        if j == i or sim[i, j] < SIMILARITY_THRESHOLD or sim[i, j] >= DUPLICATE_THRESHOLD:
            continue
        dst = url_list[j]
        if (src, dst) in existing:
            continue
        anchor, context = find_anchor(content_text[i], pages.at[j, "title"], pages.at[j, "cluster_name"] or "")
        opps.append({
            "source_url": src, "source_title": pages.at[i, "title"],
            "target_url": dst, "target_title": pages.at[j, "title"],
            "similarity": round(float(sim[i, j]), 3),
            "same_cluster": bool(pages.at[i, "cluster"] == pages.at[j, "cluster"]),
            "target_is_pillar": bool(pages.at[j, "is_pillar_candidate"]),
            "target_has_no_content_inlinks": dst in no_ctx,
            "suggested_anchor": anchor, "context": context,
        })
        found += 1
        if found >= SUGGESTIONS_PER_PAGE:
            break

opps = pd.DataFrame(opps)
if len(opps):
    opps["priority"] = (opps["similarity"]
                        + 0.10 * opps["target_is_pillar"]
                        + 0.10 * opps["target_has_no_content_inlinks"]
                        + 0.05 * (opps["suggested_anchor"] != ""))
    opps = opps.sort_values("priority", ascending=False).reset_index(drop=True)
print(f"{len(opps)} link suggestions, {int((opps['suggested_anchor'] != '').sum()) if len(opps) else 0} with an in-text anchor candidate")
opps.head(15)

## 10. Export

Download these into your audit deliverable (or a Data Studio report) so you can benchmark before and after.

In [ ]:
metrics.merge(pages[["url", "cluster", "cluster_name", "is_pillar_candidate", "centroid_sim"]], on="url", how="left") \
       .to_csv("page_link_metrics_and_clusters.csv", index=False)
links[["Source", "Destination", "Anchor", "anchor_type", "anchor_words", "Link Position", "Follow"]] \
       .to_csv("anchor_text_report.csv", index=False)
summary.to_csv("topic_clusters_summary.csv", index=False)
if len(bridges):
    bridges.to_csv("bridge_pages.csv", index=False)
duplicates.to_csv("possible_duplicates.csv", index=False)
opps.to_csv("internal_link_opportunities.csv", index=False)
print("Saved: page_link_metrics_and_clusters.csv, anchor_text_report.csv, topic_clusters_summary.csv,",
      "bridge_pages.csv, possible_duplicates.csv, internal_link_opportunities.csv, topic_clusters.png")

if "google.colab" in sys.modules:
    from google.colab import files
    for f in ["page_link_metrics_and_clusters.csv", "anchor_text_report.csv", "topic_clusters_summary.csv",
              "internal_link_opportunities.csv", "topic_clusters.png"]:
        files.download(f)

## Before you act on the suggestions

* **Read them.** Similarity is not relevance to the reader. A human decides whether a link helps someone on that page.
* **Don't add every link.** A handful of useful, well-placed contextual links beats twenty.
* **Measure.** Benchmark the metrics in section 4 now, implement in batches, and compare. Internal linking effects are hard to predict, so test.